# 19 - Train Sevilla Dish Normalization Reranker v1

Este notebook entrena el modelo de normalización/entity linking de platos para Hidden Gems Sevilla.

Objetivo:

```text
mention_text + context_sentence + candidate_dish
→ score de correspondencia
```

Este modelo no sustituye al catálogo ni a las reglas. Actúa como reranker:

detección de mención
→ generación de candidatos por catálogo/aliases
→ reranking con modelo
→ dish_id final

Modelo base:

dccuchile/bert-base-spanish-wwm-cased

Tarea:

Sequence Classification binaria:
0 = candidato incorrecto
1 = candidato correcto

Métricas principales:

accuracy@1
accuracy@3
MRR
pair_f1
pair_auc

In [ ]:
# ============================================================
# 01. Instalación e imports
# ============================================================

!pip -q install -U transformers datasets accelerate evaluate scikit-learn

import os
import re
import json
import math
import random
import shutil
import zipfile
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 02. Configuración general
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

NOTEBOOK_NAME = "19_train_sevilla_dish_normalization_reranker_v1"
VERSION = "sevilla_dish_normalization_reranker_beto_v1"

MODEL_CHECKPOINT = "dccuchile/bert-base-spanish-wwm-cased"

OUTPUT_ROOT = Path("/kaggle/working")
MODEL_OUTPUT_DIR = OUTPUT_ROOT / VERSION
REPORTS_DIR = OUTPUT_ROOT / "reports"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"

for path in [MODEL_OUTPUT_DIR, REPORTS_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
LABEL_SMOOTHING = 0.01

USE_FP16 = bool(torch.cuda.is_available())

# Negativos por mención. Más negativos = entrenamiento más fuerte, pero más lento.
MAX_NEGATIVES_PER_ROW_TRAIN = 5
MAX_NEGATIVES_PER_ROW_EVAL = 8

CONFIG = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "model_checkpoint": MODEL_CHECKPOINT,
    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "label_smoothing": LABEL_SMOOTHING,
    "max_negatives_per_row_train": MAX_NEGATIVES_PER_ROW_TRAIN,
    "max_negatives_per_row_eval": MAX_NEGATIVES_PER_ROW_EVAL,
    "seed": SEED,
    "fp16": USE_FP16,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 03. Localización automática del dataset
# ============================================================

def find_input_file(filename_patterns: List[str]) -> Path:
    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    candidates = []
    for root in search_roots:
        if root.exists():
            for pattern in filename_patterns:
                candidates.extend(root.rglob(pattern))

    if not candidates:
        raise FileNotFoundError(
            f"No se encontró ningún archivo con patrones: {filename_patterns}. "
            "Revisa que hayas añadido el dataset privado de Kaggle."
        )

    candidates = sorted(candidates, key=lambda p: len(str(p)))
    return candidates[0]


DATA_PATH = find_input_file([
    "sevilla_dish_normalization_annotation_dataset_v1_extended.jsonl",
])

print("Dataset encontrado:", DATA_PATH)

In [ ]:
# ============================================================
# 04. Carga y validación
# ============================================================

def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


if DATA_PATH.suffix.lower() == ".jsonl":
    df = read_jsonl(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head(3))
print(df.columns.tolist())

required_cols = [
    "annotation_id",
    "mention_text",
    "context_sentence",
    "candidate_dish_options_json",
    "current_dish_id",
    "current_dish_name",
    "current_dish_display_name",
    "manual_normalization_status",
    "manual_correct_dish_id",
    "manual_correct_dish_name",
    "manual_new_dish_name",
    "annotation_status",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas obligatorias: {missing}")

for col in ["mention_text", "context_sentence", "target_clause_context", "review_text_raw"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

print("annotation_status:")
display(df["annotation_status"].fillna("missing").value_counts())

print("manual_normalization_status:")
display(df["manual_normalization_status"].fillna("missing").value_counts())

In [ ]:
# ============================================================
# 05. Utilidades de normalización y parseo
# ============================================================

def strip_accents(text: str) -> str:
    text = str(text or "")
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )


def normalize_text(text: str) -> str:
    text = strip_accents(str(text or "").lower())
    text = re.sub(r"[^a-z0-9ñáéíóúü\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_missing(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    value_str = str(value).strip()
    return value_str == "" or value_str.lower() in {"nan", "none", "null", "na"}


def parse_candidate_options(value: Any) -> List[Dict[str, Any]]:
    if value is None:
        return []

    if isinstance(value, list):
        raw = value
    else:
        value_str = str(value).strip()
        if not value_str or value_str.lower() in {"nan", "none", "null"}:
            return []
        try:
            raw = json.loads(value_str)
        except Exception:
            return []

    out = []
    for item in raw:
        if not isinstance(item, dict):
            continue
        dish_id = str(item.get("dish_id", "")).strip()
        display = str(item.get("dish_display_name", "") or item.get("display_name", "") or "").strip()
        normalized = str(item.get("dish_normalized_name", "") or "").strip()
        score_hint = item.get("score_hint", 0)

        if not dish_id and not display:
            continue

        out.append({
            "dish_id": dish_id,
            "dish_display_name": display,
            "dish_normalized_name": normalized,
            "score_hint": float(score_hint) if str(score_hint).replace(".", "", 1).isdigit() else 0.0,
        })

    return out


def make_candidate(
    dish_id: Any,
    dish_name: Any,
    normalized_name: Any = "",
    score_hint: float = 999.0,
    source: str = "gold_added",
) -> Dict[str, Any]:
    dish_id = "" if is_missing(dish_id) else str(dish_id)
    dish_name = "" if is_missing(dish_name) else str(dish_name)
    normalized_name = normalize_text(dish_name if is_missing(normalized_name) else normalized_name)

    return {
        "dish_id": dish_id,
        "dish_display_name": dish_name,
        "dish_normalized_name": normalized_name,
        "score_hint": float(score_hint),
        "candidate_source": source,
    }


def candidate_key(candidate: Dict[str, Any]) -> str:
    if candidate.get("dish_id"):
        return f"id::{candidate['dish_id']}"
    return f"name::{normalize_text(candidate.get('dish_display_name', ''))}"


def dedupe_candidates(candidates: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for cand in candidates:
        key = candidate_key(cand)
        if key not in seen:
            out.append(cand)
            seen.add(key)
    return out

In [ ]:
# ============================================================
# 06. Determinar gold dish por fila
# ============================================================

def determine_gold(row: pd.Series) -> Dict[str, Any]:
    status = str(row.get("manual_normalization_status", "") or "").strip().lower()
    annotation_status = str(row.get("annotation_status", "") or "").strip().lower()

    manual_correct_id = row.get("manual_correct_dish_id")
    manual_correct_name = row.get("manual_correct_dish_name")
    manual_new_name = row.get("manual_new_dish_name")

    current_id = row.get("current_dish_id")
    current_name = row.get("current_dish_display_name")
    if is_missing(current_name):
        current_name = row.get("current_dish_name")

    # Gold manual con dish_id existente.
    if not is_missing(manual_correct_id):
        return {
            "gold_dish_id": str(manual_correct_id),
            "gold_dish_name": str(manual_correct_name) if not is_missing(manual_correct_name) else "",
            "gold_source": "manual_existing_dish",
            "is_linkable": True,
        }

    # Gold manual con plato nuevo.
    if status == "needs_new_dish" and not is_missing(manual_new_name):
        return {
            "gold_dish_id": f"NEW::{normalize_text(manual_new_name)}",
            "gold_dish_name": str(manual_new_name),
            "gold_source": "manual_new_dish",
            "is_linkable": True,
        }

    # Si está marcado como correct, el current dish es gold.
    if status == "correct":
        return {
            "gold_dish_id": str(current_id),
            "gold_dish_name": str(current_name),
            "gold_source": "manual_current_correct",
            "is_linkable": True,
        }

    # Filas pendientes: usamos current dish como weak label.
    if annotation_status == "pending" or status in {"", "nan", "missing"}:
        if not is_missing(current_id):
            return {
                "gold_dish_id": str(current_id),
                "gold_dish_name": str(current_name),
                "gold_source": "weak_current_dish",
                "is_linkable": True,
            }

    # Wrong/too_generic/unclear sin gold suficiente.
    return {
        "gold_dish_id": None,
        "gold_dish_name": None,
        "gold_source": "unusable_or_unclear",
        "is_linkable": False,
    }


gold_rows = []
for _, row in df.iterrows():
    gold_rows.append(determine_gold(row))

gold_df = pd.DataFrame(gold_rows)
df = pd.concat([df.reset_index(drop=True), gold_df], axis=1)

df["row_uid"] = [
    f"norm_row_{i:06d}_{str(aid)}"
    for i, aid in enumerate(df["annotation_id"].fillna("no_annotation").astype(str), start=1)
]

print("gold_source:")
display(df["gold_source"].value_counts())

print("is_linkable:")
display(df["is_linkable"].value_counts())

usable_df = df[df["is_linkable"] == True].copy().reset_index(drop=True)
print("Usable rows:", usable_df.shape)
display(usable_df[["row_uid", "mention_text", "current_dish_display_name", "manual_normalization_status", "gold_dish_name", "gold_source"]].head(10))

In [ ]:
# ============================================================
# 07. Preparar candidatos por fila
# ============================================================

def prepare_row_candidates(row: pd.Series) -> List[Dict[str, Any]]:
    candidates = parse_candidate_options(row.get("candidate_dish_options_json"))

    # Añadir current dish si falta.
    current_name = row.get("current_dish_display_name")
    if is_missing(current_name):
        current_name = row.get("current_dish_name")

    if not is_missing(row.get("current_dish_id")):
        candidates.append(make_candidate(
            row.get("current_dish_id"),
            current_name,
            row.get("current_dish_name"),
            score_hint=500,
            source="current_dish_added",
        ))

    # Añadir gold si falta.
    if row.get("is_linkable") and not is_missing(row.get("gold_dish_id")):
        candidates.append(make_candidate(
            row.get("gold_dish_id"),
            row.get("gold_dish_name"),
            row.get("gold_dish_name"),
            score_hint=1000,
            source="gold_added",
        ))

    candidates = dedupe_candidates(candidates)

    # Ordenar por hint descendente para que los negativos sean duros.
    candidates = sorted(candidates, key=lambda c: c.get("score_hint", 0), reverse=True)

    return candidates


usable_df["candidate_options"] = usable_df.apply(prepare_row_candidates, axis=1)
usable_df["candidate_count"] = usable_df["candidate_options"].map(len)

print("Candidate count summary:")
display(usable_df["candidate_count"].describe())

print("Rows without candidates:", int((usable_df["candidate_count"] == 0).sum()))

usable_df = usable_df[usable_df["candidate_count"] > 0].copy().reset_index(drop=True)

display(usable_df[["row_uid", "mention_text", "gold_dish_name", "gold_source", "candidate_count", "candidate_options"]].head(3))

In [ ]:
# ============================================================
# 08. Split por fila, no por pares
# ============================================================

usable_df["split_group"] = (
    usable_df["gold_source"].astype(str)
    + "__"
    + usable_df["manual_normalization_status"].fillna("missing").astype(str)
)

group_counts = usable_df["split_group"].value_counts()
rare_groups = set(group_counts[group_counts < 3].index)
usable_df["split_group_safe"] = usable_df["split_group"].where(
    ~usable_df["split_group"].isin(rare_groups),
    "rare"
)

train_rows, temp_rows = train_test_split(
    usable_df,
    test_size=0.20,
    random_state=SEED,
    stratify=usable_df["split_group_safe"],
)

temp_counts = temp_rows["split_group_safe"].value_counts()
if (temp_counts < 2).any():
    val_rows, test_rows = train_test_split(
        temp_rows,
        test_size=0.50,
        random_state=SEED,
    )
else:
    val_rows, test_rows = train_test_split(
        temp_rows,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_rows["split_group_safe"],
    )

train_rows = train_rows.reset_index(drop=True)
val_rows = val_rows.reset_index(drop=True)
test_rows = test_rows.reset_index(drop=True)

split_summary = {
    "train_rows": int(len(train_rows)),
    "validation_rows": int(len(val_rows)),
    "test_rows": int(len(test_rows)),
    "train_gold_source": train_rows["gold_source"].value_counts().to_dict(),
    "validation_gold_source": val_rows["gold_source"].value_counts().to_dict(),
    "test_gold_source": test_rows["gold_source"].value_counts().to_dict(),
}

print(json.dumps(split_summary, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 09. Construcción de pares positivos/negativos
# ============================================================

def build_query_text(row: pd.Series) -> str:
    mention = str(row.get("mention_text", "") or "").strip()
    context = str(row.get("context_sentence", "") or "").strip()
    target_context = str(row.get("target_clause_context", "") or "").strip()
    place = str(row.get("place_name", "") or "").strip()

    # Preferimos contexto de frase; target_clause sirve como apoyo si existe.
    parts = [
        f"Mención: {mention}",
    ]

    if context:
        parts.append(f"Contexto: {context}")

    if target_context and target_context != context:
        parts.append(f"Fragmento: {target_context}")

    if place:
        parts.append(f"Local: {place}")

    return " | ".join(parts)


def build_candidate_text(candidate: Dict[str, Any]) -> str:
    display = candidate.get("dish_display_name", "") or ""
    normalized = candidate.get("dish_normalized_name", "") or ""

    if normalized and normalize_text(display) != normalize_text(normalized):
        return f"Plato candidato: {display} | Nombre normalizado: {normalized}"

    return f"Plato candidato: {display}"


def is_gold_candidate(candidate: Dict[str, Any], row: pd.Series) -> bool:
    gold_id = str(row.get("gold_dish_id", "") or "")
    gold_name = normalize_text(row.get("gold_dish_name", ""))

    cand_id = str(candidate.get("dish_id", "") or "")
    cand_name = normalize_text(candidate.get("dish_display_name", ""))

    if gold_id and cand_id and gold_id == cand_id:
        return True

    if gold_name and cand_name and gold_name == cand_name:
        return True

    return False


def source_weight(row: pd.Series) -> float:
    gold_source = str(row.get("gold_source", ""))
    status = str(row.get("manual_normalization_status", "") or "")

    if gold_source.startswith("manual"):
        return 2.0
    if status in {"wrong", "too_generic", "needs_new_dish"}:
        return 2.5
    if gold_source == "weak_current_dish":
        return 0.75
    return 1.0


def build_pair_records(
    row_df: pd.DataFrame,
    split_name: str,
    max_negatives: int,
) -> pd.DataFrame:
    records = []

    rng = random.Random(SEED)

    for _, row in row_df.iterrows():
        candidates = row["candidate_options"]
        positives = [c for c in candidates if is_gold_candidate(c, row)]
        negatives = [c for c in candidates if not is_gold_candidate(c, row)]

        if not positives:
            continue

        # Un positivo principal.
        pos = positives[0]

        query_text = build_query_text(row)

        base_meta = {
            "row_uid": row["row_uid"],
            "annotation_id": row.get("annotation_id"),
            "split": split_name,
            "mention_text": row.get("mention_text"),
            "gold_dish_id": row.get("gold_dish_id"),
            "gold_dish_name": row.get("gold_dish_name"),
            "gold_source": row.get("gold_source"),
            "manual_normalization_status": row.get("manual_normalization_status"),
            "place_name": row.get("place_name"),
            "review_id": row.get("review_id"),
        }

        records.append({
            **base_meta,
            "text_a": query_text,
            "text_b": build_candidate_text(pos),
            "candidate_dish_id": pos.get("dish_id"),
            "candidate_dish_name": pos.get("dish_display_name"),
            "candidate_score_hint": pos.get("score_hint", 0),
            "label": 1,
            "sample_weight": source_weight(row),
        })

        # Negativos duros: candidatos con score_hint más alto.
        negatives = sorted(negatives, key=lambda c: c.get("score_hint", 0), reverse=True)

        if len(negatives) > max_negatives:
            # Mezcla: los mejores hints + algo aleatorio.
            hard = negatives[: max_negatives // 2]
            rest = negatives[max_negatives // 2 :]
            extra_n = max_negatives - len(hard)
            sampled = rng.sample(rest, min(extra_n, len(rest))) if rest else []
            selected_negatives = hard + sampled
        else:
            selected_negatives = negatives

        for neg in selected_negatives:
            records.append({
                **base_meta,
                "text_a": query_text,
                "text_b": build_candidate_text(neg),
                "candidate_dish_id": neg.get("dish_id"),
                "candidate_dish_name": neg.get("dish_display_name"),
                "candidate_score_hint": neg.get("score_hint", 0),
                "label": 0,
                "sample_weight": source_weight(row),
            })

    return pd.DataFrame(records)


train_pairs = build_pair_records(train_rows, "train", MAX_NEGATIVES_PER_ROW_TRAIN)
val_pairs = build_pair_records(val_rows, "validation", MAX_NEGATIVES_PER_ROW_EVAL)
test_pairs = build_pair_records(test_rows, "test", MAX_NEGATIVES_PER_ROW_EVAL)

print("Train pairs:", train_pairs.shape)
print("Validation pairs:", val_pairs.shape)
print("Test pairs:", test_pairs.shape)

print("Train label distribution:")
display(train_pairs["label"].value_counts())

print("Validation label distribution:")
display(val_pairs["label"].value_counts())

display(train_pairs.head(5))

In [ ]:
# ============================================================
# 10. Conversión a Dataset y tokenización
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

def pairs_to_records(pair_df: pd.DataFrame) -> List[Dict[str, Any]]:
    records = []
    for _, row in pair_df.iterrows():
        records.append({
            "text_a": str(row["text_a"]),
            "text_b": str(row["text_b"]),
            "label": int(row["label"]),
            "sample_weight": float(row["sample_weight"]),
            "row_uid": str(row["row_uid"]),
            "candidate_dish_id": str(row["candidate_dish_id"]),
            "candidate_dish_name": str(row["candidate_dish_name"]),
        })
    return records


dataset = DatasetDict({
    "train": Dataset.from_list(pairs_to_records(train_pairs)),
    "validation": Dataset.from_list(pairs_to_records(val_pairs)),
    "test": Dataset.from_list(pairs_to_records(test_pairs)),
})


def tokenize_pairs(batch: Dict[str, List[Any]]) -> Dict[str, Any]:
    tokenized = tokenizer(
        batch["text_a"],
        batch["text_b"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    tokenized["labels"] = batch["label"]
    tokenized["sample_weight"] = batch["sample_weight"]

    return tokenized


tokenized_dataset = dataset.map(
    tokenize_pairs,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_dataset

In [ ]:
# ============================================================
# 11. Métricas de clasificación por pares
# ============================================================

def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(logits)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def compute_pair_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)[:, 1]
    preds = (probs >= 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0,
    )

    acc = accuracy_score(labels, preds)

    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = 0.0

    try:
        auprc = average_precision_score(labels, probs)
    except Exception:
        auprc = 0.0

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "auprc": auprc,
    }

In [ ]:
# ============================================================
# 12. Trainer con sample weights
# ============================================================

class WeightedPairTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        sample_weight = inputs.pop("sample_weight", None)

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            reduction="none",
            label_smoothing=LABEL_SMOOTHING,
        )

        losses = loss_fct(logits, labels)

        if sample_weight is not None:
            sample_weight = sample_weight.to(losses.device).float()
            losses = losses * sample_weight

        loss = losses.mean()

        return (loss, outputs) if return_outputs else loss

In [ ]:
# ============================================================
# 13. Modelo y TrainingArguments
# ============================================================

model_config = AutoConfig.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label={0: "not_match", 1: "match"},
    label2id={"not_match": 0, "match": 1},
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    config=model_config,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

steps_per_epoch = math.ceil(
    len(tokenized_dataset["train"]) / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
)
total_steps = steps_per_epoch * NUM_EPOCHS
WARMUP_STEPS = int(total_steps * WARMUP_RATIO)

print("steps_per_epoch:", steps_per_epoch)
print("total_steps:", total_steps)
print("warmup_steps:", WARMUP_STEPS)

def build_training_args():
    common_kwargs = dict(
        output_dir=str(MODEL_OUTPUT_DIR),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        logging_steps=25,
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="auprc",
        greater_is_better=True,
        fp16=USE_FP16,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
    )

    try:
        return TrainingArguments(
            eval_strategy="epoch",
            **common_kwargs,
        )
    except TypeError:
        return TrainingArguments(
            evaluation_strategy="epoch",
            **common_kwargs,
        )

training_args = build_training_args()

trainer = WeightedPairTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_pair_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(training_args)

In [ ]:
# ============================================================
# 14. Entrenamiento
# ============================================================

train_result = trainer.train()

train_metrics = train_result.metrics

trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

with (REPORTS_DIR / "train_metrics_v1.json").open("w", encoding="utf-8") as f:
    json.dump(train_metrics, f, indent=2, ensure_ascii=False)

print(json.dumps(train_metrics, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 15. Evaluación por pares
# ============================================================

validation_pair_metrics = trainer.evaluate(tokenized_dataset["validation"])
test_pair_metrics = trainer.evaluate(tokenized_dataset["test"])

print("Validation pair metrics:")
print(json.dumps(validation_pair_metrics, indent=2, ensure_ascii=False))

print("\nTest pair metrics:")
print(json.dumps(test_pair_metrics, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 16. Evaluación ranking por mención
# ============================================================

def predict_pair_scores(pair_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    ds = Dataset.from_list(pairs_to_records(pair_df))
    tok = ds.map(
        tokenize_pairs,
        batched=True,
        remove_columns=ds.column_names,
    )

    output = trainer.predict(tok)
    probs = softmax_np(output.predictions)[:, 1]

    out = pair_df.copy()
    out["match_score"] = probs
    return out


def ranking_metrics(scored_df: pd.DataFrame) -> Dict[str, Any]:
    rows = []
    group_metrics = []

    for row_uid, group in scored_df.groupby("row_uid"):
        group = group.sort_values("match_score", ascending=False).reset_index(drop=True)

        labels = group["label"].astype(int).tolist()
        has_positive = any(v == 1 for v in labels)

        if not has_positive:
            continue

        positive_positions = [i + 1 for i, v in enumerate(labels) if v == 1]
        best_pos = min(positive_positions)

        acc1 = 1 if best_pos == 1 else 0
        acc3 = 1 if best_pos <= 3 else 0
        mrr = 1.0 / best_pos

        group_metrics.append({
            "row_uid": row_uid,
            "candidate_count": len(group),
            "rank_of_gold": best_pos,
            "accuracy_at_1": acc1,
            "accuracy_at_3": acc3,
            "mrr": mrr,
            "top_candidate_name": group.iloc[0]["candidate_dish_name"],
            "gold_dish_name": group[group["label"] == 1].iloc[0]["gold_dish_name"],
            "top_score": float(group.iloc[0]["match_score"]),
            "gold_score": float(group[group["label"] == 1].iloc[0]["match_score"]),
        })

    metrics_df = pd.DataFrame(group_metrics)

    if len(metrics_df) == 0:
        return {
            "rows": 0,
            "accuracy_at_1": 0.0,
            "accuracy_at_3": 0.0,
            "mrr": 0.0,
            "metrics_df": metrics_df,
        }

    return {
        "rows": int(len(metrics_df)),
        "accuracy_at_1": float(metrics_df["accuracy_at_1"].mean()),
        "accuracy_at_3": float(metrics_df["accuracy_at_3"].mean()),
        "mrr": float(metrics_df["mrr"].mean()),
        "metrics_df": metrics_df,
    }


val_scored_pairs = predict_pair_scores(val_pairs, "validation")
test_scored_pairs = predict_pair_scores(test_pairs, "test")

val_ranking = ranking_metrics(val_scored_pairs)
test_ranking = ranking_metrics(test_scored_pairs)

val_ranking_df = val_ranking.pop("metrics_df")
test_ranking_df = test_ranking.pop("metrics_df")

val_ranking_df.to_csv(PREDICTIONS_DIR / "normalization_val_ranking_results_v1.csv", index=False, encoding="utf-8")
test_ranking_df.to_csv(PREDICTIONS_DIR / "normalization_test_ranking_results_v1.csv", index=False, encoding="utf-8")

val_scored_pairs.to_csv(PREDICTIONS_DIR / "normalization_val_scored_pairs_v1.csv", index=False, encoding="utf-8")
test_scored_pairs.to_csv(PREDICTIONS_DIR / "normalization_test_scored_pairs_v1.csv", index=False, encoding="utf-8")

print("Validation ranking:")
print(json.dumps(val_ranking, indent=2, ensure_ascii=False))

print("\nTest ranking:")
print(json.dumps(test_ranking, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 17. Análisis de errores de ranking
# ============================================================

def build_ranking_error_analysis(scored_df: pd.DataFrame, base_rows_df: pd.DataFrame) -> pd.DataFrame:
    errors = []

    row_meta = base_rows_df.set_index("row_uid").to_dict(orient="index")

    for row_uid, group in scored_df.groupby("row_uid"):
        group = group.sort_values("match_score", ascending=False).reset_index(drop=True)

        if int(group.iloc[0]["label"]) == 1:
            continue

        gold_rows = group[group["label"] == 1]
        if len(gold_rows) == 0:
            continue

        gold_row = gold_rows.iloc[0]
        top_row = group.iloc[0]

        meta = row_meta.get(row_uid, {})

        errors.append({
            "row_uid": row_uid,
            "annotation_id": meta.get("annotation_id"),
            "mention_text": meta.get("mention_text"),
            "context_sentence": meta.get("context_sentence"),
            "manual_normalization_status": meta.get("manual_normalization_status"),
            "gold_source": meta.get("gold_source"),
            "gold_dish_name": gold_row["candidate_dish_name"],
            "gold_score": gold_row["match_score"],
            "predicted_dish_name": top_row["candidate_dish_name"],
            "predicted_score": top_row["match_score"],
            "score_margin_pred_minus_gold": float(top_row["match_score"] - gold_row["match_score"]),
            "candidate_count": len(group),
        })

    return pd.DataFrame(errors)


val_error_df = build_ranking_error_analysis(val_scored_pairs, val_rows)
test_error_df = build_ranking_error_analysis(test_scored_pairs, test_rows)

val_error_df.to_csv(PREDICTIONS_DIR / "normalization_val_error_analysis_v1.csv", index=False, encoding="utf-8")
test_error_df.to_csv(PREDICTIONS_DIR / "normalization_test_error_analysis_v1.csv", index=False, encoding="utf-8")

print("Validation ranking errors:", val_error_df.shape)
print("Test ranking errors:", test_error_df.shape)

display(test_error_df.head(30))

In [ ]:
# ============================================================
# 18. Función de inferencia manual
# ============================================================

def score_candidates_for_mention(
    mention_text: str,
    context_sentence: str,
    candidate_names: List[str],
    place_name: str = "",
) -> pd.DataFrame:
    query_text = " | ".join([
        f"Mención: {mention_text}",
        f"Contexto: {context_sentence}",
        f"Local: {place_name}" if place_name else "",
    ]).strip(" | ")

    records = []
    for idx, name in enumerate(candidate_names):
        records.append({
            "text_a": query_text,
            "text_b": f"Plato candidato: {name}",
            "label": 0,
            "sample_weight": 1.0,
            "row_uid": "manual_example",
            "candidate_dish_id": f"candidate_{idx}",
            "candidate_dish_name": name,
        })

    ds = Dataset.from_list(records)
    tok = ds.map(tokenize_pairs, batched=True, remove_columns=ds.column_names)
    output = trainer.predict(tok)
    scores = softmax_np(output.predictions)[:, 1]

    result = pd.DataFrame({
        "candidate_dish_name": candidate_names,
        "match_score": scores,
    }).sort_values("match_score", ascending=False)

    return result


examples_result = score_candidates_for_mention(
    mention_text="ensaladilla de gambas",
    context_sentence="Pedimos ensaladilla de gambas y croquetas de jamón, ambas espectaculares.",
    candidate_names=[
        "ensaladilla",
        "gambas",
        "ensaladilla de gambas",
        "croquetas de jamón",
        "salmorejo",
    ],
)

display(examples_result)

In [ ]:
# ============================================================
# 19. Guardado de resumen final
# ============================================================

final_summary = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model_checkpoint": MODEL_CHECKPOINT,
    "dataset_path": str(DATA_PATH),
    "dataset_rows_raw": int(len(df)),
    "usable_rows": int(len(usable_df)),
    "pair_counts": {
        "train_pairs": int(len(train_pairs)),
        "validation_pairs": int(len(val_pairs)),
        "test_pairs": int(len(test_pairs)),
    },
    "split_summary": split_summary,
    "train_metrics": train_metrics,
    "pair_metrics": {
        "validation": validation_pair_metrics,
        "test": test_pair_metrics,
    },
    "ranking_metrics": {
        "validation": val_ranking,
        "test": test_ranking,
    },
    "outputs": {
        "model_dir": str(MODEL_OUTPUT_DIR),
        "val_ranking_results": str(PREDICTIONS_DIR / "normalization_val_ranking_results_v1.csv"),
        "test_ranking_results": str(PREDICTIONS_DIR / "normalization_test_ranking_results_v1.csv"),
        "val_error_analysis": str(PREDICTIONS_DIR / "normalization_val_error_analysis_v1.csv"),
        "test_error_analysis": str(PREDICTIONS_DIR / "normalization_test_error_analysis_v1.csv"),
    },
    "notes": [
        "This model is a cross-encoder reranker for dish normalization/entity linking.",
        "It scores mention/context against candidate dish names.",
        "It should be used after candidate generation, not as a closed catalog classifier.",
        "Rows with needs_new_dish can be partially learned, but production should still support manual/new dish fallback.",
    ],
}

with (REPORTS_DIR / "normalization_training_summary_v1.json").open("w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2, ensure_ascii=False)

print(json.dumps(final_summary, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 20. Crear ZIP descargable del modelo
# ============================================================

zip_path = OUTPUT_ROOT / f"{VERSION}.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in MODEL_OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(MODEL_OUTPUT_DIR.parent))

    for file_path in REPORTS_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(OUTPUT_ROOT))

    for file_path in PREDICTIONS_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(OUTPUT_ROOT))

print("ZIP creado:", zip_path)
print("Tamaño MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))